In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
DATA_PATH = "book_bestseller_clean.csv"

df_books = pd.read_csv(

    DATA_PATH,

    encoding="utf-8-sig",

)

print("데이터 크기:", df_books.shape)

print("컬럼:", df_books.columns.tolist())

print("상품명 결측치:", df_books["상품명"].isna().sum())

df_books[["상품명"]].head(10)

데이터 크기: (199, 8)
컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']
상품명 결측치: 0


,상품명
0,소년이 온다
1,모순
2,결국 국민이 합니다
3,혼모노
4,급류
5,초역 부처의 말
6,청춘의 독서(특별증보판)
7,어른의 행복은 조용하다
8,채식주의자
9,단 한 번의 삶(강물에디션 활판인쇄 한정판)


In [3]:
df_reco = df_books.copy()

df_reco["상품명"] = (
    df_reco["상품명"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df_reco = (
    df_reco[df_reco["상품명"] != ""]
    .reset_index(drop=True)
)
print("추천에 사용할 도서 수:", len(df_reco))

추천에 사용할 도서 수: 199


# [실습 4] 콘텐츠 기반 추천(Content-Based Recommendation) 이해하기

## 1. 핵심 개념 및 작동 원리
- **개념**: 선택한 항목의 특징을 분석하여 이와 유사한 특징을 가진 다른 항목을 추천하는 방식
- **특징 추출 대상**: 도서의 **상품명(제목) 텍스트**
- **유사도 계산 과정**:
  1. **도서 A 제목** $\rightarrow$ **TF-IDF 벡터 A** 변환
  2. **도서 B 제목** $\rightarrow$ **TF-IDF 벡터 B** 변환
  3. 두 벡터 간 유사도(코사인 유사도 등) 측정
  4. 유사도가 높을 경우 **추천 후보**로 선정

---

## 2. 활용 정보 및 한계점
### ❌ 사용하지 않는 정보
- 사용자 구매 이력
- 클릭 이력
- 평점 및 판매량
- 독자 취향

### 💡 특징
- 사용자의 행동 데이터나 취향을 고려하지 않으므로 **개인화 추천**이라기보다는 **'도서 제목 기반 유사 도서 추천'**에 해당함

# [실습 5] 코사인 유사도(Cosine Similarity) 이해하기

## 1. 핵심 개념
- **개념**: 두 벡터 간의 **각도**를 이용해 방향이 얼마나 비슷한지(유사한지) 측정하는 지표
- **값의 의미**:
  - **1에 가까움**: 방향이 매우 비슷함 (현재 표현에서 유사도가 높음)
  - **0에 가까움**: 직교에 가까우며 공통 특징이 거의 없음

---

## 2. 예시 분석
```python
vectors = np.array([
    [1, 1], # Vector 0
    [2, 2], # Vector 1
    [1, 0]  # Vector 2
])

## 실습 6. 실제 도서 제목을 TF-IDF로 변환하기

In [4]:
# [실습 6] 실제 도서 제목을 TF-IDF로 변환하기
## 1. 실습 코드
titles = df_reco["상품명"]
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(titles)

print("도서 수:", tfidf_matrix.shape[0])
print("단어 수:", tfidf_matrix.shape[1])

도서 수: 199
단어 수: 536


## 📌 실습 6 결과 요약

- **전체 분석 도서 수**: 199권
- **추출된 총 단어 수**: 536개
- **생성된 행렬 형태 (Shape)**: `(199, 536)`

> **핵심 요약**: 199개의 도서 제목을 분석하여 총 536개의 단어(어휘)를 추출하였으며, 이에 따라 **199행 536열** 규모의 TF-IDF 행렬이 생성되었습니다.

## 실습 7. 기준 도서 선택하기

In [5]:
# [실습 7] 기준 도서 선택하기

selected_index = 0
selected_title = df_reco.loc[
    selected_index,
    "상품명",
]

print("선택 도서:", selected_title)

선택 도서: 소년이 온다


## 📌 실습 7 결과 요약

- **선택된 인덱스 (selected_index)**: `0`
- **선택된 기준 도서명**: `소년이 온다`

> **핵심 요약**: 인덱스 `0`번 위치의 실제 도서명을 확인한 결과, 기준 도서로 **"소년이 온다"**가 선택되었습니다.

## 실습 8. 선택 도서와 전체 도서의 유사도 계산하기

In [7]:
# [실습 8] 선택 도서와 전체 도서의 유사도 계산하기
from sklearn.metrics.pairwise import cosine_similarity

# 선택한 도서의 벡터 가져오기
selected_vector = tfidf_matrix[selected_index]

# 전체 도서와의 코사인 유사도 계산
similarity_scores = cosine_similarity(
    selected_vector,
    tfidf_matrix,
).flatten()

# 결과 출력 및 도서 수 일치 여부 확인
print("유사도 개수:", len(similarity_scores))
print("전체 도서 수:", len(df_reco))

# 인덱스 대응 확인 (0번, 1번 인덱스 예시)
print("\n--- 인덱스 대응 확인 ---")
print("similarity_scores[0]:", similarity_scores[0], "↔ df_reco.iloc[0]['상품명']:", df_reco.iloc[0]["상품명"])
print("similarity_scores[1]:", similarity_scores[1], "↔ df_reco.iloc[1]['상품명']:", df_reco.iloc[1]["상품명"])

유사도 개수: 199
전체 도서 수: 199

--- 인덱스 대응 확인 ---
similarity_scores[0]: 1.0000000000000002 ↔ df_reco.iloc[0]['상품명']: 소년이 온다
similarity_scores[1]: 0.0 ↔ df_reco.iloc[1]['상품명']: 모순


## 📌 실습 8 결과 요약

- **계산된 유사도 개수**: 199개[cite: 4]
- **전체 도서 수**: 199권[cite: 4]
- **인덱스 대응 확인**:
  - `0번 인덱스 ("소년이 온다")`: 자기 자신과의 유사도 **1.0** (자기 자신)[cite: 4]
  - `1번 인덱스 ("모순")`: 유사도 **0.0** (공통 단어 없음)[cite: 4]

> **핵심 요약**: 선택한 기준 도서("소년이 온다")와 전체 도서 199권의 유사도를 계산한 결과, 자기 자신과의 유사도는 **1.0**으로 정확히 대칭되며, 공통 단어가 없는 "모순"과는 **0.0**의 유사도를 보여 각 점수와 도서가 1:1로 일치함을 확인했습니다[cite: 4].

## 실습 9. 유사도가 높은 순서 확인하기

In [9]:
score_df = pd.DataFrame({
    "index": np.arange(len(df_reco)),
    "상품명": df_reco["상품명"],
    "similarity": similarity_scores,
})

score_df.sort_values(
    "similarity",
    ascending=False,
).head(10)

,index,상품명,similarity
0,0,소년이 온다,1.0
1,1,모순,0.0
2,2,결국 국민이 합니다,0.0
3,3,혼모노,0.0
4,4,급류,0.0
5,5,초역 부처의 말,0.0
6,6,청춘의 독서(특별증보판),0.0
7,7,어른의 행복은 조용하다,0.0
8,8,채식주의자,0.0
9,9,단 한 번의 삶(강물에디션 활판인쇄 한정판),0.0


## 📌 실습 9 결과 요약

- **유사도 1위 (index 0)**: `소년이 온다` (similarity: **1.0**, 자기 자신)[cite: 5]
- **상위 2~10위 도서**: `모순`, `결국 국민이 합니다`, `혼모노` 등 (similarity: 모두 **0.0**)[cite: 5]
> **핵심 요약**: 기준 도서("소년이 온다")와 유사도가 가장 높은 상위 10개 항목을 정렬한 결과, **자기 자신(유사도 1.0)**을 제외한 나머지 모든 도서의 유사도가 **0.0**으로 측정되었습니다[cite: 5]. 이는 제목 텍스트 기반 추천 시 공통된 단어(어휘)가 전혀 존재하지 않아 발생하는 현상입니다[cite: 5].

## 실습 10. 자기 자신을 제외하고 Top 5 만들기

In [10]:
# [실습 10] 자기 자신을 제외하고 Top 5 만들기

# 유사도가 높은 순서의 index 구하기
sorted_indices = similarity_scores.argsort()[::-1]

# 자기 자신을 제외하고 상위 5개 선택
recommended_indices = [
    idx
    for idx in sorted_indices
    if idx != selected_index
][:5]

# 추천 결과 데이터프레임 생성 및 유사도 추가
result = df_reco.loc[
    recommended_indices,
    ["상품명"],
].copy()

result["similarity"] = [
    round(float(similarity_scores[idx]), 4)
    for idx in recommended_indices
]

result

,상품명,similarity
197,당신에게 분명 좋은 일만 생길 거예요,0.0
198,결핍은 우리를 어떻게 변화시키는가,0.0
195,당신은 결국 무엇이든 해내는 사람(특별 리커버 에디션),0.0
194,용의자 X의 헌신,0.0
193,미치도록 보고 싶었던 돈의 얼굴,0.0


## 📌 실습 10 결과 요약

- **자기 자신 제외 여부**: 완료 (`selected_index = 0` 제외됨)[cite: 6]
- **최종 추출 개수**: 상위 4~5개 도서 추출 (출력 화면상 index 197, 198, 195, 194 등 확인)[cite: 6]
- **추천 도서 목록 및 유사도**:
  - `당신에게 분명 좋은 일만 생길 거예요`: **0.0**[cite: 6]
  - `결핍은 우리를 어떻게 변화시키는가`: **0.0**[cite: 6]
  - `당신은 결국 무엇이든 해내는 사람(특별 리커버 에디션)`: **0.0**[cite: 6]
  - `용의자 X의 헌신`: **0.0**[cite: 6]

> **핵심 요약**: 자기 자신을 정렬 목록에서 제외하고 상위 도서를 추출했으나, 기준 도서("소년이 온다")와 제목상 공통 단어를 가진 도서가 카탈로그 내에 존재하지 않아 상위 추천 후보들의 유사도가 모두 **0.0**으로 측정되었습니다[cite: 6].

In [11]:
# [실습 11] 추천 결과에 메타데이터 추가하기

# 존재 유무를 확인하여 사용할 컬럼 목록 추출
available_columns = [
    column
    for column in [
        "상품명",
        "인물",
        "출판사",
        "분야",
    ]
    if column in df_reco.columns
]

# 선택된 추천 항목에 메타데이터 정보 추가
result = df_reco.loc[
    recommended_indices,
    available_columns,
].copy()

# 유사도 점수 컬럼 추가 (소수점 4자리 반영)
result["similarity"] = [
    round(float(similarity_scores[idx]), 4)
    for idx in recommended_indices
]

# 선택 도서와 추천 결과 비교 출력
print("선택 도서:")
print(selected_title)
print("\n추천 도서:")
display(result)

선택 도서:
소년이 온다

추천 도서:


,상품명,출판사,분야,similarity
197,당신에게 분명 좋은 일만 생길 거예요,다담북스,시/에세이,0.0
198,결핍은 우리를 어떻게 변화시키는가,빌리버튼,경제/경영,0.0
195,당신은 결국 무엇이든 해내는 사람(특별 리커버 에디션),필름(Feelm),시/에세이,0.0
194,용의자 X의 헌신,재인,소설,0.0
193,미치도록 보고 싶었던 돈의 얼굴,영진닷컴,경제/경영,0.0


## 📌 실습 11 결과 요약

- **선택 기준 도서**: `소년이 온다`[cite: 7]
- **추가된 메타데이터 컬럼**: `출판사`, `분야`[cite: 7]
- **추천 도서 비교 및 검토**:
  - `당신에게 분명 좋은 일만 생길 거예요` (다담북스 / 시/에세이 / **similarity: 0.0**)[cite: 7]
  - `결핍은 우리를 어떻게 변화시키는가` (빌리버튼 / 경제/경영 / **similarity: 0.0**)[cite: 7]
  - `당신은 결국 무엇이든 해내는 사람(특별 리커버 에디션)` (필름(Feelm) / 시/에세이 / **similarity: 0.0**)[cite: 7]
  - `용의자 X의 헌신` (재인 / 소설 / **similarity: 0.0**)[cite: 7]
  - `미치도록 보고 싶었던 돈의 얼굴` (영진닷컴 / 경제/경영 / **similarity: 0.0**)[cite: 7]

> **핵심 요약**: 메타데이터(출판사, 분야)를 함께 조회하여 비교한 결과, 공통 단어가 존재하지 않아 추천 후보들의 유사도가 모두 **0.0**일 뿐만 아니라 분야(시/에세이, 경제/경영, 소설 등) 역시 일관성이 없는 도서들이 나열되었음을 확인할 수 있습니다[cite: 7]. 단어(제목) 기반 코사인 유사도 방식의 한계점과 추가적인 메타데이터 활용의 필요성을 보여줍니다[cite: 7].

In [12]:
# [실습 12] 추천 함수 만들기
from sklearn.metrics.pairwise import cosine_similarity


def recommend_books(
    selected_index,
    df,
    tfidf_matrix,
    top_n=5,
):
    if selected_index not in df.index:
        raise IndexError(f"유효하지 않은 index입니다: {selected_index}")

    selected_title = df.loc[
        selected_index,
        "상품명",
    ]

    # 유사도 계산
    similarity_scores = cosine_similarity(
        tfidf_matrix[selected_index],
        tfidf_matrix,
    ).flatten()

    # 유사도 높은 순 정렬
    sorted_indices = similarity_scores.argsort()[::-1]

    # 자기 자신 및 동일한 제목 제외 후 Top N 선택
    recommended_indices = []
    for idx in sorted_indices:
        if idx == selected_index:
            continue
        if df.loc[idx, "상품명"] == selected_title:
            continue
        recommended_indices.append(idx)
        if len(recommended_indices) >= top_n:
            break

    # 존재 유무 확인 후 컬럼 추출
    columns = [
        column
        for column in [
            "상품명",
            "인물",
            "출판사",
            "분야",
        ]
        if column in df.columns
    ]

    # 결과 데이터프레임 생성 및 반환
    result = df.loc[
        recommended_indices,
        columns,
    ].copy()

    result["similarity"] = [
        round(float(similarity_scores[idx]), 4)
        for idx in recommended_indices
    ]

    return result.reset_index(drop=True)

In [13]:
# [실습 13] 추천 함수 실행하기

selected_index = 0
print(
    "선택 도서:",
    df_reco.loc[selected_index, "상품명"],
)

# 추천 함수 실행 (Top 5 추출)
recommendations = recommend_books(
    selected_index=selected_index,
    df=df_reco,
    tfidf_matrix=tfidf_matrix,
    top_n=5,
)

# 검증: 데이터프레임과 TF-IDF 행렬의 행 수 일치 확인
assert df_reco.shape[0] == tfidf_matrix.shape[0]

# 추천 결과 출력
recommendations

선택 도서: 소년이 온다


,상품명,출판사,분야,similarity
0,당신에게 분명 좋은 일만 생길 거예요,다담북스,시/에세이,0.0
1,결핍은 우리를 어떻게 변화시키는가,빌리버튼,경제/경영,0.0
2,당신은 결국 무엇이든 해내는 사람(특별 리커버 에디션),필름(Feelm),시/에세이,0.0
3,용의자 X의 헌신,재인,소설,0.0
4,미치도록 보고 싶었던 돈의 얼굴,영진닷컴,경제/경영,0.0


## 📌 실습 13 결과 요약

- **선택 기준 도서**: `소년이 온다`[cite: 8]
- **행 수 검증 (`assert`)**: 정상 통과 (오류 없이 실행 완료)[cite: 8]
- **추천 함수 실행 결과**:
  - `0`: `당신에게 분명 좋은 일만 생길 거예요` (다담북스 / 시/에세이 / **similarity: 0.0**)[cite: 8]
  - `1`: `결핍은 우리를 어떻게 변화시키는가` (빌리버튼 / 경제/경영 / **similarity: 0.0**)[cite: 8]
  - `2`: `당신은 결국 무엇이든 해내는 사람(특별 리커버 에디션)` (필름(Feelm) / 시/에세이 / **similarity: 0.0**)[cite: 8]
  - `3`: `용의자 X의 헌신` (재인 / 소설 / **similarity: 0.0**)[cite: 8]
  - `4`: `미치도록 보고 싶었던 돈의 얼굴` (영진닷컴 / 경제/경영 / **similarity: 0.0**)[cite: 8]

> **핵심 요약**: 작성한 `recommend_books` 함수가 정상 작동하여 자기 자신을 제외한 **Top 5 도서 목록**과 인덱스가 초기화된 데이터프레임을 정상 반환했습니다[cite: 8]. 다만, 기준 도서와 텍스트상 유사한 도서가 없어 반환된 추천 결과의 유사도는 모두 **0.0**입니다[cite: 8].

In [14]:
# [실습 14] 추천 결과 저장하기 및 확인

# 추천 결과를 CSV 파일로 저장 (한글 깨짐 방지를 위해 utf-8-sig 사용)
recommendations.to_csv(
    "chapter05_recommendations.csv",
    index=False,
    encoding="utf-8-sig",
)

# 저장된 CSV 파일을 다시 읽어와서 정상 저장 여부 확인
pd.read_csv(
    "chapter05_recommendations.csv",
    encoding="utf-8-sig",
)

,상품명,출판사,분야,similarity
0,당신에게 분명 좋은 일만 생길 거예요,다담북스,시/에세이,0.0
1,결핍은 우리를 어떻게 변화시키는가,빌리버튼,경제/경영,0.0
2,당신은 결국 무엇이든 해내는 사람(특별 리커버 에디션),필름(Feelm),시/에세이,0.0
3,용의자 X의 헌신,재인,소설,0.0
4,미치도록 보고 싶었던 돈의 얼굴,영진닷컴,경제/경영,0.0


# [실습 15] 추천 결과의 한계 이해하기

## 1. 상품명(제목) 텍스트 기반 추천의 한계
- **제목 단어의 불일치**: 같은 분야의 도서라도 제목에 공통 단어가 없으면 유사도가 낮게 측정됨
- **표현의 우연성**: 다른 분야의 도서라도 제목에 공통 표현이 많으면 유사도가 높게 측정될 수 있음

## 2. 코사인 유사도(Cosine Similarity)의 개념적 한계
- **유사도가 높음**은 단지 **텍스트 표현이 유사함**을 의미함
- 다음과 같은 의미로 해석하지 않도록 주의가 필요함:
  - ❌ 사용자가 반드시 좋아함
  - ❌ 더 좋은 품질의 책
  - ❌ 구매 가능성이 높은 책

## 3. 올바른 결과 해석 및 설명 방식
- **올바른 표현**: "현재 TF-IDF 제목 표현 기준으로 유사도가 높은 도서입니다."
- **피해야 할 표현**: "사용자가 가장 좋아할 책입니다."